# Paul Graham Essay Feeds — regenerate in Colab

Unofficial **metadata-only** multi-format feed regenerator for the official essays index:

- Source: https://paulgraham.com/articles.html
- Outputs: RSS 2.0, Atom 1.0, JSON Feed 1.1, OPML 2.0

> **Disclaimer:** This is an unofficial project. It is not affiliated with or endorsed by Paul Graham.

### What this notebook does

1. Installs [uv](https://docs.astral.sh/uv/) and Python **3.13** (required by the package).
2. Clones (or reuses) the repository and installs the zero-runtime-dependency package.
3. Regenerates all feeds either from the **live index** or the **bundled fixture**.
4. Validates offline with `pg-essay-feeds check`.
5. Optionally zips the artifacts for download.

### Fetch stack

Network fetches use the **Python standard library** (`urllib.request`) only — not `httpx`, `requests`, or `trafilatura`.

## 1. Configuration

Edit the cell below, then run all cells (**Runtime → Run all**).

In [ ]:
from pathlib import Path

# --- user settings ---------------------------------------------------------

# Git clone URL of this repository (or a fork).
REPO_URL = "https://github.com/wyattowalsh/paul-graham-essay-feeds.git"

# Branch or tag to check out.
REPO_REF = "main"

# Public base URL used for self-links and OPML (must be a real absolute URL).
PUBLIC_BASE_URL = "https://paul-graham-essay-feeds.vercel.app/"

# "live"  → fetch https://paulgraham.com/articles.html
# Offline: pass --source-file yourself; notebook uses live by default.
SOURCE_MODE = "live"  # or "fixture"

# Force rewrite even when logical content is unchanged.
FORCE = False

# Local workspace under /content in Colab (or cwd when run elsewhere).
WORK_ROOT = Path("/content/pg-essay-feeds-work")

# ---------------------------------------------------------------------------

assert SOURCE_MODE in {"live", "fixture"}, "SOURCE_MODE must be 'live' or 'fixture'"
assert PUBLIC_BASE_URL.startswith(("http://", "https://")), "PUBLIC_BASE_URL must be absolute"
print("Config OK")
print(f"  SOURCE_MODE     = {SOURCE_MODE}")
print(f"  PUBLIC_BASE_URL = {PUBLIC_BASE_URL}")
print(f"  REPO_URL        = {REPO_URL}@{REPO_REF}")
print(f"  WORK_ROOT       = {WORK_ROOT}")


## 2. Install uv + Python 3.13

Colab’s default interpreter is often older than 3.13. The project requires **Python ≥ 3.13**, so we install a managed toolchain with `uv`.

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path


def run(cmd, **kwargs):
    print("+", " ".join(str(c) for c in cmd))
    subprocess.run(cmd, check=True, **kwargs)


# Install uv if missing.
if shutil.which("uv") is None:
    run([sys.executable, "-m", "pip", "install", "-q", "uv"])

uv = shutil.which("uv") or str(Path.home() / ".local/bin/uv")
assert Path(uv).exists() or shutil.which("uv"), "uv not found after install"

run(["uv", "--version"])
run(["uv", "python", "install", "3.13"])
print("uv and Python 3.13 ready")

## 3. Clone repository and locate package root

Expects a top-level package checkout (repo root contains `pyproject.toml`).


In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

WORK_ROOT = Path(WORK_ROOT)
WORK_ROOT.mkdir(parents=True, exist_ok=True)
CLONE_DIR = WORK_ROOT / "repo"


def run(cmd, **kwargs):
    print("+", " ".join(str(c) for c in cmd))
    subprocess.run(cmd, check=True, **kwargs)


if not (CLONE_DIR / ".git").exists():
    if CLONE_DIR.exists():
        shutil.rmtree(CLONE_DIR)
    run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(CLONE_DIR)])
else:
    run(["git", "-C", str(CLONE_DIR), "fetch", "--depth", "1", "origin", REPO_REF])
    run(["git", "-C", str(CLONE_DIR), "checkout", REPO_REF])
    run(["git", "-C", str(CLONE_DIR), "pull", "--ff-only", "origin", REPO_REF])


def find_package_root(start: Path) -> Path:
    """Find directory that contains pyproject.toml for this package."""
    direct = start / "pyproject.toml"
    if direct.is_file() and "paul-graham-essay-feeds" in direct.read_text(
        encoding="utf-8", errors="ignore"
    ):
        return start
    # Walk one level for other nestings.
    for child in start.iterdir():
        candidate = child / "pyproject.toml"
        if candidate.is_file() and "paul-graham-essay-feeds" in candidate.read_text(
            encoding="utf-8", errors="ignore"
        ):
            return child
    raise FileNotFoundError(
        f"Could not locate package pyproject.toml under {start}. "
        "Set REPO_URL to a checkout that contains the project."
    )


PKG_ROOT = find_package_root(CLONE_DIR)
os.chdir(PKG_ROOT)
print("Package root:", PKG_ROOT)
print("Contents:", sorted(p.name for p in PKG_ROOT.iterdir())[:20])

## 4. Install the package (Python 3.13 via uv)

In [ ]:
import os
import subprocess
from pathlib import Path

os.chdir(PKG_ROOT)


def run(cmd, **kwargs):
    print("+", " ".join(str(c) for c in cmd))
    subprocess.run(cmd, check=True, **kwargs)


# Use project Python 3.13 and install the package + console script.
run(["uv", "sync", "--python", "3.13", "--all-groups"])
run(["uv", "run", "--python", "3.13", "pg-essay-feeds", "--version"])
print("Package installed")

## 5. Regenerate feeds

- **live**: conditional HTTP fetch of the official index (stdlib `urllib`).
- **fixture**: deterministic rebuild from the audited HTML snapshot (no live HTML required after clone).

In [ ]:
import os
import subprocess
from pathlib import Path

os.chdir(PKG_ROOT)

cmd = [
    "uv",
    "run",
    "--python",
    "3.13",
    "pg-essay-feeds",
    "update",
    "--public-base-url",
    PUBLIC_BASE_URL,
    "--repo-root",
    str(PKG_ROOT),
]

if SOURCE_MODE == "fixture":
    raise SystemExit("Notebook uses live source only; use CLI --source-file for offline HTML")
    assert fixture.is_file(), f"Missing fixture: {fixture}"
    cmd.extend(["--source-file", str(fixture)])
    # Fixture rebuilds are usually forced so lastBuildDate / outputs refresh cleanly.
    cmd.append("--force")
elif FORCE:
    cmd.append("--force")

print("+", " ".join(cmd))
result = subprocess.run(cmd, check=False)
if result.returncode != 0:
    raise SystemExit(
        f"pg-essay-feeds update failed with exit code {result.returncode}. "
        "If reconciliation rejected a source change, review with --allow-removals / "
        "--allow-nonprefix-additions only after manual inspection."
    )
print("Update finished")


## 6. Offline validation + summary

In [ ]:
import json
import os
import subprocess
import xml.etree.ElementTree as ET
from pathlib import Path

os.chdir(PKG_ROOT)

check = subprocess.run(
    [
        "uv",
        "run",
        "--python",
        "3.13",
        "pg-essay-feeds",
        "check",
        "--public-base-url",
        PUBLIC_BASE_URL,
        "--repo-root",
        str(PKG_ROOT),
    ],
    check=False,
)
if check.returncode != 0:
    raise SystemExit(f"check failed with exit code {check.returncode}")

rss_path = PKG_ROOT / "feeds" / "rss.xml"
atom_path = PKG_ROOT / "feeds" / "atom.xml"
json_path = PKG_ROOT / "feeds" / "feed.json"
opml_path = PKG_ROOT / "feeds" / "subscriptions.opml"
report_path = PKG_ROOT / "reports" / "validation.json"

for path in (rss_path, atom_path, json_path, opml_path, report_path):
    assert path.is_file(), f"Missing artifact: {path}"

rss_root = ET.fromstring(rss_path.read_bytes())
rss_items = rss_root.find("channel").findall("item")
first = rss_items[0]
last = rss_items[-1]

atom_root = ET.fromstring(atom_path.read_bytes())
atom_entries = [el for el in list(atom_root) if el.tag.endswith("entry")]

feed_json = json.loads(json_path.read_text(encoding="utf-8"))
opml_root = ET.fromstring(opml_path.read_bytes())
opml_outlines = opml_root.find("body").findall("outline")
report = json.loads(report_path.read_text(encoding="utf-8"))

print(
    "Validation:", "VALID" if report.get("valid") else "INVALID", f"status={report.get('status')}"
)
print("RSS items:", len(rss_items))
print("Atom entries:", len(atom_entries))
print("JSON Feed items:", len(feed_json.get("items", [])), "version=", feed_json.get("version"))
print("OPML outlines:", len(opml_outlines))
print("First:", first.findtext("title"), "→", first.findtext("link"))
print("Last:", last.findtext("title"), "→", last.findtext("link"))
print()
print("Artifacts:")
for path in (rss_path, atom_path, json_path, opml_path, report_path, PKG_ROOT / "SHA256SUMS"):
    if path.is_file():
        print(f"  {path.relative_to(PKG_ROOT)}  ({path.stat().st_size} bytes)")

## 7. Download zip of generated feeds (Colab)

Creates `pg-essay-feeds-artifacts.zip` and triggers a browser download when running in Google Colab.

In [ ]:
import shutil
import zipfile
from pathlib import Path

zip_path = WORK_ROOT / "pg-essay-feeds-artifacts.zip"
if zip_path.exists():
    zip_path.unlink()

members = [
    PKG_ROOT / "feeds" / "rss.xml",
    PKG_ROOT / "feeds" / "atom.xml",
    PKG_ROOT / "feeds" / "feed.json",
    PKG_ROOT / "feeds" / "subscriptions.opml",
    PKG_ROOT / "data" / "essays.json",
    PKG_ROOT / "data" / "state.json",
    PKG_ROOT / "reports" / "validation.json",
    PKG_ROOT / "SHA256SUMS",
]

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in members:
        if path.is_file():
            zf.write(path, arcname=path.relative_to(PKG_ROOT).as_posix())

print("Wrote", zip_path, f"({zip_path.stat().st_size} bytes)")

try:
    from google.colab import files  # type: ignore

    files.download(str(zip_path))
    print("Colab download started")
except Exception as exc:
    print("Not in Colab or download helper unavailable:", exc)
    print("Zip path:", zip_path)

## Safety notes

- Newest-prefix additions are accepted automatically.
- Removals, retained-item reordering, and mid-history insertions **fail closed** by default.
- Reviewed overrides exist (`--allow-removals`, `--allow-nonprefix-additions`, `--min-items`) but should only be used after inspecting the source page and the proposed diff.
- Atom `updated` values are **feed-observation metadata**, not original publication dates.
- Never invent public/self URLs: set `PUBLIC_BASE_URL` to a real deployment base only.